# PubMedQA Sample Data Explorer

This notebook keeps only the dataset loading and sample manipulation code from `0521_intro2genai_demo_batched.ipynb`. It does not run model calls or create WTT cache artifacts.

In [1]:
# Run this once if the dependencies are missing.
# %pip install -q datasets pandas

from typing import Any, Dict, List

import pandas as pd
from datasets import load_dataset
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 180)

/home/ubuntu/.pyenv/versions/cau-2026-intro2genai-demo/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data Loading Helpers

These functions are copied from the original notebook and trimmed to the PubMedQA sample path.

In [2]:
def context_to_text(raw_context: Any) -> str:
    if raw_context is None:
        return ""
    if isinstance(raw_context, str):
        return raw_context.strip()
    if isinstance(raw_context, list):
        return "\n".join(str(x).strip() for x in raw_context if str(x).strip())
    if isinstance(raw_context, dict):
        contexts = raw_context.get("contexts") or raw_context.get("context") or raw_context.get("sentences") or []
        labels = raw_context.get("labels") or []
        meshes = raw_context.get("meshes") or []
        lines = []
        for i, sent in enumerate(contexts):
            sent = str(sent).strip()
            if not sent:
                continue
            label = str(labels[i]).strip() if i < len(labels) else ""
            lines.append(f"[{label}] {sent}" if label else sent)
        if meshes:
            mesh_text = ", ".join(str(x).strip() for x in meshes if str(x).strip())
            if mesh_text:
                lines.append(f"MeSH terms: {mesh_text}")
        return "\n".join(lines).strip()
    return str(raw_context).strip()


def extract_gold_answers(raw: Dict[str, Any]) -> List[str]:
    answers = []
    for item in [
        raw.get("final_decision") or raw.get("answer") or raw.get("label"),
        raw.get("long_answer") or raw.get("long_ans") or raw.get("explanation"),
    ]:
        if item is None:
            continue
        if isinstance(item, list):
            answers.extend(str(x).strip() for x in item if str(x).strip())
        elif str(item).strip():
            answers.append(str(item).strip())
    return answers


def load_pubmedqa_sample(dataset: Any, split: str = "train", idx: int = 0) -> Dict[str, Any]:
    if split not in dataset:
        split = list(dataset.keys())[0]

    raw = dataset[split][idx]
    question = raw.get("question") or raw.get("query") or raw.get("prompt") or ""
    context = context_to_text(raw.get("context") or raw.get("abstract") or raw.get("passage"))
    title = raw.get("title") or raw.get("pubid") or raw.get("id") or ""
    sample_id = raw.get("pubid") or raw.get("id") or raw.get("qid") or f"{split}-{idx}"
    gold_label = str(raw.get("final_decision") or raw.get("answer") or raw.get("label") or "").strip().lower()
    return {
        "idx": idx,
        "id": str(sample_id),
        "title": str(title).strip(),
        "question": str(question).strip(),
        "context": context.strip(),
        "gold_answers": extract_gold_answers(raw),
        "gold_label": gold_label,
    }


def select_reasonable_pubmedqa_samples(
    dataset: Any,
    split: str = "train",
    max_context_chars: int = 4000,
    n: int = 100,
    prefer_labeled: bool = True,
) -> List[Dict[str, Any]]:
    if split not in dataset:
        split = list(dataset.keys())[0]

    selected = []
    fallback = []
    for idx in range(len(dataset[split])):
        sample = load_pubmedqa_sample(dataset, split, idx)
        if not sample["question"] or not sample["context"]:
            continue
        if len(sample["context"]) > max_context_chars:
            continue
        if sample["gold_label"] in {"yes", "no", "maybe"}:
            selected.append(sample)
        else:
            fallback.append(sample)
        if prefer_labeled and len(selected) >= n:
            break
        if not prefer_labeled and len(selected) + len(fallback) >= n:
            break
    return selected[:n] if prefer_labeled and selected else (selected + fallback)[:n]


def load_candidate_samples(n_samples: int, max_context_chars: int) -> List[Dict[str, Any]]:
    dataset = load_dataset("qiaojin/PubMedQA", "pqa_labeled")
    split = list(dataset.keys())[0]
    samples = select_reasonable_pubmedqa_samples(
        dataset,
        split=split,
        max_context_chars=max_context_chars,
        n=n_samples,
    )
    print(f"Selected {len(samples)} PubMedQA samples from split={split}")
    return samples

## Load The Raw Dataset

In [3]:
DATASET_NAME = "qiaojin/PubMedQA"
CONFIG_NAME = "pqa_labeled"

dataset = load_dataset(DATASET_NAME, CONFIG_NAME)
split = list(dataset.keys())[0]

print(f"Dataset: {DATASET_NAME}/{CONFIG_NAME}")
print(f"Splits: {list(dataset.keys())}")
print(f"Using split: {split}")
print(f"Rows: {len(dataset[split])}")
dataset

Dataset: qiaojin/PubMedQA/pqa_labeled
Splits: ['train']
Using split: train
Rows: 1000


DatasetDict({
    train: Dataset({
        features: ['pubid', 'question', 'context', 'long_answer', 'final_decision'],
        num_rows: 1000
    })
})

In [4]:
dataset[split].features

{'pubid': Value('int32'),
 'question': Value('string'),
 'context': {'contexts': List(Value('string')),
  'labels': List(Value('string')),
  'meshes': List(Value('string')),
  'reasoning_required_pred': List(Value('string')),
  'reasoning_free_pred': List(Value('string'))},
 'long_answer': Value('string'),
 'final_decision': Value('string')}

In [5]:
raw_preview = dataset[split].select(range(min(5, len(dataset[split])))).to_pandas()
display(raw_preview)

,pubid,question,context,long_answer,final_decision
0,21645374,Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?,{'contexts': ['Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves ...,"Results depicted mitochondrial dynamics in vivo as PCD progresses within the lace plant, and highlight the correlation of this organelle with other organelles during developmen...",yes
1,16418930,Landolt C and snellen e acuity: differences in strabismus amblyopia?,{'contexts': ['Assessment of visual acuity depends on the optotypes used for measurement. The ability to recognize different optotypes differs even if their critical details ap...,"Using the charts described, there was only a slight overestimation of visual acuity by the Snellen E compared to the Landolt C, even in strabismus amblyopia. Small differences ...",no
2,9488747,"Syncope during bathing in infants, a pediatric form of water-induced urticaria?",{'contexts': ['Apparent life-threatening events in infants are a difficult and frequent problem in pediatric practice. The prognosis is uncertain because of risk of sudden infa...,"""Aquagenic maladies"" could be a pediatric form of the aquagenic urticaria.",yes
3,17208539,Are the long-term results of the transanal pull-through equal to those of the transabdominal pull-through?,"{'contexts': ['The transanal endorectal pull-through (TERPT) is becoming the most popular procedure in the treatment of Hirschsprung disease (HD), but overstretching of the ana...",Our long-term study showed significantly better (2-fold) results regarding the continence score for the abdominal approach compared with the transanal pull-through. The stool p...,no
4,10808977,Can tailored interventions increase mammography use among HMO women?,"{'contexts': ['Telephone counseling and tailored print communications have emerged as promising methods for promoting mammography screening. However, there has been little rese...","The effects of the intervention were most pronounced after the first intervention. Compared to usual care, telephone counseling seemed particularly effective at promoting chang...",yes


In [22]:
import json
# json.loads(raw_preview[['context']].iloc[0].iloc)
raw_preview[['context']].iloc[0].iloc[0]

{'contexts': array(['Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants.',
        'The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), cells in early stages of PCD (EPCD), and cells in late stages of PCD (LPCD). Window stage leaves were stained with the mitochondri

## Normalize And Select Samples

In [25]:
def samples_to_dataframe(samples: List[Dict[str, Any]]) -> pd.DataFrame:
    df = pd.DataFrame(samples)
    if df.empty:
        return df
    df["context_chars"] = df["context"].str.len()
    df["question_chars"] = df["question"].str.len()
    columns = [
        "idx",
        "id",
        "gold_label",
        "question_chars",
        "context_chars",
        "title",
        "question",
        "gold_answers",
        "context",
    ]
    return df[columns]


N_SAMPLES = 20
MAX_CONTEXT_CHARS = 1000

samples = select_reasonable_pubmedqa_samples(
    dataset,
    split=split,
    max_context_chars=MAX_CONTEXT_CHARS,
    n=N_SAMPLES,
)
samples_df = samples_to_dataframe(samples)

display(samples_df)

,idx,id,gold_label,question_chars,context_chars,title,question,gold_answers,context
0,8,17113061,no,80,899,17113061,Do mutations causing low HDL-C promote increased carotid intima-media thickness?,"[no, Genetic variants identified in the present study may be insufficient to promote early carotid atherosclerosis.]","[BACKGROUND] Although observational data support an inverse relationship between high-density lipoprotein (HDL) cholesterol and coronary heart disease (CHD), genetic HDL defici..."
1,17,17096624,yes,72,870,17096624,Do patterns of knowledge and attitudes exist among unvaccinated seniors?,"[yes, Findings suggest that cluster analyses may be useful in identifying groups for targeted health messages.]",[OBJECTIVE] To examine patterns of knowledge and attitudes among adults aged>65 years unvaccinated for influenza.\n[METHODS] Surveyed Medicare beneficiaries in 5 areas; cluster...
2,32,9645785,no,76,880,9645785,Is a mandatory general surgery rotation necessary in the surgical clerkship?,"[no, Effective undergraduate surgical education can be offered in many specialty settings. Removal of the requirement for general surgery in clerkship may lead to a more effect...",[BACKGROUND] Changes in the spectrum of general surgery and the delivery of surgical care have placed the requirement for a mandatory general surgery rotation in the surgical c...
3,54,25277731,maybe,76,695,25277731,Sternal fracture in growing children : A rare and often overlooked fracture?,"[maybe, Isolated sternal fractures in childhood are often due to typical age-related traumatic incidents. Ultrasonography is a useful diagnostic tool for fracture detection and...","[BACKGROUND] Sternal fractures in childhood are rare. The aim of the study was to investigate the accident mechanism, the detection of radiological and sonographical criteria a..."
4,59,27491658,yes,72,823,27491658,Can predilatation in transcatheter aortic valve implantation be omitted?,"[yes, TAVI can be performed safely without balloon predilatation and with the same early results as achieved with the standard procedure including balloon predilatation. The re...",[BACKGROUND] The use of a balloon expandable stent valve includes balloon predilatation of the aortic stenosis before valve deployment. The aim of the study was to see whether ...
5,71,18049437,yes,81,902,18049437,Is there any relationship between streptococcal infection and multiple sclerosis?,"[yes, These findings indicate that a relationship between multiple sclerosis and streptococcal infections may exist, but to acquire a better understanding of the role of group ...",[BACKGROUND] Multiple sclerosis (MS) is an immune-mediated inflammatory demyelinating disease of uncertain etiology. Although the mechanisms of inducting autoimmunity by some o...
6,86,11570976,maybe,22,951,11570976,Is it Crohn's disease?,"[maybe, Granulomatous myelotoxicity and enteritis developed in a 21 year old female within 3 weeks of initiating sulfasalazine for rheumatoid arthritis. Following a short cours...",[BACKGROUND] Sulfasalazine is a widely used anti-inflammatory agent in the treatment of inflammatory bowel disease and several rheumatological disorders. Although as many as 20...
7,95,23076787,no,77,998,23076787,Can increases in the cigarette tax rate be linked to cigarette retail prices?,"[no, Numerous studies have found that taxation is one of the most effective policy instruments for tobacco control. However, these findings come from countries that have market...",[OBJECTIVE] To explain China's cigarette pricing mechanism and the role of the Chinese State Tobacco Monopoly Administration (STMA) on cigarette pricing and taxation.\n[METHODS...
8,118,19106867,yes,80,947,19106867,"The Main Gate Syndrome: a new format in mass-casualty victim ""surge"" management?","[yes, Suicide bombing in crowded locations near an evacuation hospital may overwhelm the medical resources of the receiving center. It has been referred to as ""The Main Gate Sy...",[INTRODUCT

In [ ]:
samples_df[["gold_label", "context_chars", "question_chars"]].describe(include="all")

In [ ]:
samples_df["gold_label"].value_counts(dropna=False)

## Inspect One Sample

In [26]:
def _md_safe(value: Any) -> str:
    return str(value).replace("`", "\\`")


def show_sample(sample_number: int = 0, max_context_chars: int | None = None) -> None:
    sample = samples[sample_number]
    context = sample["context"]
    if max_context_chars is not None and len(context) > max_context_chars:
        context = context[:max_context_chars].rstrip() + "\n..."

    display(Markdown(
        f"### Sample {sample_number} / dataset idx {sample['idx']}\n\n"
        f"**ID:** `{_md_safe(sample['id'])}`  \n"
        f"**Gold label:** `{_md_safe(sample['gold_label'])}`  \n"
        f"**Question:** {_md_safe(sample['question'])}\n\n"
        f"**Gold answers:** `{_md_safe(sample['gold_answers'])}`\n\n"
        f"**Context chars:** `{len(sample['context'])}`\n\n"
        f"```text\n{context}\n```"
    ))


show_sample(7)

### Sample 7 / dataset idx 95

**ID:** `23076787`  
**Gold label:** `no`  
**Question:** Can increases in the cigarette tax rate be linked to cigarette retail prices?

**Gold answers:** `['no', "Numerous studies have found that taxation is one of the most effective policy instruments for tobacco control. However, these findings come from countries that have market economies where market forces determine prices and influence how cigarette taxes are passed to the consumers in retail prices. China's tobacco industry is not a market economy; therefore, non-market forces and the current Chinese tobacco monopoly system determine cigarette prices. The result is that tax increases do not necessarily get passed on to the retail price."]`

**Context chars:** `998`

```text
[OBJECTIVE] To explain China's cigarette pricing mechanism and the role of the Chinese State Tobacco Monopoly Administration (STMA) on cigarette pricing and taxation.
[METHODS] Published government tobacco tax documentation and statistics published by the Chinese STMA are used to analyse the interrelations among industry profits, taxes and retail price of cigarettes in China.
[RESULTS] The 2009 excise tax increase on cigarettes in China has not translated into higher retail prices because the Chinese STMA used its policy authority to ensure that retail cigarette prices did not change. The government tax increase is being collected at both the producer and wholesale levels. As a result, the 2009 excise tax increase in China has resulted in higher tax revenue for the government and lower profits for the tobacco industry, with no increase in the retail price of cigarettes for consumers.
MeSH terms: China, Commerce, Government Regulation, Humans, Taxes, Tobacco Industry, Tobacco Products
```

## Small Manipulation Examples

In [ ]:
short_yes_samples = samples_df.query("gold_label == 'yes' and context_chars <= 2500")
display(short_yes_samples)

In [ ]:
sample_number = 0
dataset_idx = int(samples_df.iloc[sample_number]["idx"])

raw_sample = dataset[split][dataset_idx]
normalized_sample = load_pubmedqa_sample(dataset, split, dataset_idx)

print("Raw sample keys:", list(raw_sample.keys()))
display(pd.DataFrame([normalized_sample]))
raw_sample